<a href="https://colab.research.google.com/github/janpfrang-hash/Steam-Iron-plotter/blob/main/Steam_Iron_bench_data_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim:**
# Automatically sort and plot and evaluate certain data from the Steam Iron Test bench

**Basic requirements**

* shall read .XLSX or should - .TDMS files
  * allow upload of 2 .xlsx files
* shall create automated diagrams with
  * consumption of water / cycle
  * steam rate / cycle
  * unique identifier - QTR number
* shall create tables with the values of
  * consumption of water / cycle
  * steam rate / cycle
* should contain a PDF export of plots and tables
* should contain a .xlsx export for the data tables
* should contain an UI with selectors to
  * plot consumption of water / cycle for single or all devices in one plot
  * plot steam rate / cycle for singe or all devices in one plot
  * enter thresholds for both values

In [ ]:
# @title
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'ipywidgets', 'openpyxl'], check=True)

import io, re, warnings, urllib.parse
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as pdf_backend
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

warnings.filterwarnings('ignore')

# ── Constants ────────────────────────────────────────────────────────────────
DUT_SHEET_PREFIX = 'Test Data Overview - DUT'
MAX_DUTS  = 10
COL_TIME  = 4
COL_WATER = 11
COL_STEAM = 13
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
          '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']

# ── State ────────────────────────────────────────────────────────────────────
state = {'dut_data': {}, 'loaded_files': [], 'job_ids': []}

# ── Helpers ──────────────────────────────────────────────────────────────────
def time_to_minutes(t):
    try:
        if hasattr(t, 'total_seconds'):
            return t.total_seconds() / 60.0
        parts = str(t).split(':')
        if len(parts) == 3:
            return int(parts[0]) * 60 + int(parts[1]) + int(parts[2]) / 60
        elif len(parts) == 2:
            return int(parts[0]) * 60 + int(parts[1])
    except Exception:
        pass
    return None

def time_to_full_hours(t):
    try:
        if hasattr(t, 'total_seconds'):
            total_min = t.total_seconds() / 60.0
        else:
            parts = str(t).split(':')
            if len(parts) == 3:
                total_min = int(parts[0]) * 60 + int(parts[1]) + int(parts[2]) / 60
            elif len(parts) == 2:
                total_min = int(parts[0]) * 60 + int(parts[1])
            else:
                return str(t)
        return f'{round(total_min / 60)}h'
    except Exception:
        return str(t)

def extract_job_id(sheet_names):
    pattern = re.compile(r'[A-Z]{3}\d{7}')
    for name in sheet_names:
        m = pattern.search(name)
        if m:
            return m.group(0)
    return ''

def load_dut_sheets_from_bytes(file_bytes, file_label):
    xl       = pd.ExcelFile(io.BytesIO(file_bytes))
    job_id   = extract_job_id(xl.sheet_names)
    dut_sheets = sorted(
        [s for s in xl.sheet_names
         if s.startswith(DUT_SHEET_PREFIX) and
         s[len(DUT_SHEET_PREFIX):].isdigit() and
         int(s[len(DUT_SHEET_PREFIX):]) <= MAX_DUTS],
        key=lambda s: int(s[len(DUT_SHEET_PREFIX):])
    )
    result = {}
    for sheet in dut_sheets:
        raw = pd.read_excel(io.BytesIO(file_bytes), sheet_name=sheet, header=None)
        df  = raw.iloc[1:, [COL_TIME, COL_WATER, COL_STEAM]].copy()
        df.columns = ['Test time', 'Water_L', 'Steam_gMin']
        df = df.dropna(subset=['Test time', 'Water_L', 'Steam_gMin'])
        df['Time_min']   = df['Test time'].apply(time_to_minutes)
        df['Water_L']    = pd.to_numeric(df['Water_L'],    errors='coerce')
        df['Steam_gMin'] = pd.to_numeric(df['Steam_gMin'], errors='coerce')
        df = df.dropna().reset_index(drop=True)
        df['Cycle'] = range(1, len(df) + 1)
        dut_key = f'{file_label}·DUT{sheet[len(DUT_SHEET_PREFIX):]}'
        result[dut_key] = df
    return job_id, result

# ── Full cycle table (1 decimal place for water & steam) ─────────────────────
def make_full_cycle_table():
    if not state['dut_data']:
        return pd.DataFrame()
    frames = []
    for dut, df in state['dut_data'].items():
        sub = df[['Cycle', 'Test time', 'Water_L', 'Steam_gMin']].copy()
        sub['Water_L']    = sub['Water_L'].round(1)
        sub['Steam_gMin'] = sub['Steam_gMin'].round(1)
        sub = sub.rename(columns={
            'Test time':  f'{dut} — Test time',
            'Water_L':    f'{dut} — Water [L]',
            'Steam_gMin': f'{dut} — Steam [g/min]',
        })
        frames.append(sub.set_index('Cycle'))
    merged = frames[0]
    for f in frames[1:]:
        merged = merged.join(f, how='outer')
    return merged.reset_index().sort_values('Cycle')

# ── Plot builder ──────────────────────────────────────────────────────────────
def build_plot(duts_to_plot, col, ylabel, title_metric, thresh_min, thresh_max,
               fig=None, ax=None):
    job_str   = '  |  ' + ', '.join(state['job_ids']) if state['job_ids'] else ''
    files_str = ', '.join(state['loaded_files'])
    title     = f"{title_metric}{job_str}  |  {files_str}"

    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=(14, 5))

    ref_df = state['dut_data'][duts_to_plot[0]]

    for i, dut in enumerate(duts_to_plot):
        df = state['dut_data'][dut]
        ax.plot(df['Cycle'], df[col].values,
                marker='o', markersize=3, linewidth=1.5,
                color=COLORS[i % len(COLORS)], label=dut)

    ax.axhline(thresh_min, color='red',    linestyle='--', linewidth=1.2,
               label=f'Min threshold ({thresh_min})')
    ax.axhline(thresh_max, color='orange', linestyle='--', linewidth=1.2,
               label=f'Max threshold ({thresh_max})')
    ax.axhspan(thresh_min, thresh_max, alpha=0.06, color='green')

    ax.set_xlabel('Cycle #', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)

    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    n_cycles = len(ref_df)
    step     = max(1, n_cycles // 10)
    tick_idx = list(range(0, n_cycles, step))
    if (n_cycles - 1) not in tick_idx:
        tick_idx.append(n_cycles - 1)
    ax2.set_xticks(ref_df['Cycle'].iloc[tick_idx].values)
    ax2.set_xticklabels(
        [time_to_full_hours(ref_df['Test time'].iloc[i]) for i in tick_idx],
        rotation=30, ha='left', fontsize=8
    )
    ax2.set_xlabel(f'Test time [{duts_to_plot[0]}]', fontsize=9)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=28)
    ax.legend(fontsize=9, loc='best', ncol=2)
    ax.grid(True, linestyle=':', alpha=0.5)
    fig.tight_layout()
    return fig, ax

def generate_pdf_bytes():
    all_duts = list(state['dut_data'].keys())
    buf = io.BytesIO()
    with pdf_backend.PdfPages(buf) as pdf:
        fig1, _ = build_plot(
            all_duts, 'Water_L', 'Water Consumption per Cycle [L]',
            'Water Consumption per Cycle',
            w_thresh_water_min.value, w_thresh_water_max.value)
        pdf.savefig(fig1, bbox_inches='tight')
        plt.close(fig1)
        fig2, _ = build_plot(
            all_duts, 'Steam_gMin', 'Steam Rate per Cycle [g/min]',
            'Steam Rate per Cycle',
            w_thresh_steam_min.value, w_thresh_steam_max.value)
        pdf.savefig(fig2, bbox_inches='tight')
        plt.close(fig2)
    buf.seek(0)
    return buf.read()

def pdf_filename():
    job_part = '_'.join(state['job_ids']) if state['job_ids'] else 'export'
    return f"{job_part}_report.pdf"

# ── Widgets ──────────────────────────────────────────────────────────────────
style   = {'description_width': '200px'}
lay_btn = widgets.Layout(width='340px', height='40px')
lay_num = widgets.Layout(width='260px')

w_upload = widgets.FileUpload(
    accept='.xlsx', multiple=True,
    description='📂 Select XLSX files',
    layout=widgets.Layout(width='320px')
)
w_progress       = widgets.IntProgress(value=0, min=0, max=100, description='0%',
                                        bar_style='info',
                                        layout=widgets.Layout(width='400px', visibility='hidden'))
w_progress_label = widgets.Label(value='', layout=widgets.Layout(visibility='hidden'))
w_file_label     = widgets.Label(value='No file loaded.')

w_dut_select = widgets.SelectMultiple(
    options=[], description='Devices (DUTs)',
    rows=8, layout=widgets.Layout(width='260px'), style=style
)
w_select_all_water = widgets.Checkbox(value=True, description='Select all DUTs', indent=False)
w_select_all_steam = widgets.Checkbox(value=True, description='Select all DUTs', indent=False)

w_thresh_water_min = widgets.FloatText(value=0.80, description='Water min [L]:',     style=style, layout=lay_num)
w_thresh_water_max = widgets.FloatText(value=1.20, description='Water max [L]:',     style=style, layout=lay_num)
w_thresh_steam_min = widgets.FloatText(value=14.0, description='Steam min [g/min]:', style=style, layout=lay_num)
w_thresh_steam_max = widgets.FloatText(value=22.0, description='Steam max [g/min]:', style=style, layout=lay_num)

w_btn_water = widgets.Button(description='📊  PLOT WATER CONSUMPTION / CYCLE',
                              button_style='primary', layout=lay_btn, disabled=True)
w_btn_steam = widgets.Button(description='📊  PLOT STEAM RATE / CYCLE',
                              button_style='success', layout=lay_btn, disabled=True)
w_btn_table = widgets.Button(description='📥  DOWNLOAD SUMMARY TABLE AS .XLSX',
                              button_style='warning', layout=lay_btn, disabled=True)
w_btn_pdf   = widgets.Button(description='📄  EXPORT PLOT AS PDF',
                              button_style='danger',  layout=lay_btn, disabled=True)

out_status = widgets.Output()
out_plot   = widgets.Output()
out_table  = widgets.Output()

# ── Callbacks ────────────────────────────────────────────────────────────────
def on_upload(change):
    if not w_upload.value:
        return
    uploaded = w_upload.value
    n_files  = len(uploaded)
    if n_files == 0:
        return

    w_progress.layout.visibility       = 'visible'
    w_progress_label.layout.visibility = 'visible'
    w_progress.value       = 0
    w_progress.bar_style   = 'info'
    w_progress.description = '0%'
    w_progress_label.value = f'Processing 0 / {n_files} file(s) …'

    state['dut_data']     = {}
    state['loaded_files'] = []
    state['job_ids']      = []

    errors = []
    for idx, (fname, info) in enumerate(uploaded.items()):
        step_pct = int((idx / n_files) * 100)
        w_progress.value       = step_pct
        w_progress.description = f'{step_pct}%'
        w_progress_label.value = f'Reading {fname} ({idx + 1}/{n_files}) …'
        try:
            job_id, dut_dict = load_dut_sheets_from_bytes(info['content'], f'F{idx+1}')
            if job_id and job_id not in state['job_ids']:
                state['job_ids'].append(job_id)
            state['loaded_files'].append(fname)
            state['dut_data'].update(dut_dict)
        except Exception as e:
            errors.append(f'{fname}: {e}')

    w_progress.value       = 100
    w_progress.description = '100%'
    w_progress.bar_style   = 'danger' if errors else 'success'
    w_progress_label.value = (
        f'✅  Done — {n_files} file(s) loaded.' if not errors
        else f'⚠️  Finished with errors: {"; ".join(errors)}'
    )

    dut_names = list(state['dut_data'].keys())
    w_dut_select.options = dut_names
    w_dut_select.value   = dut_names

    job_str = '  |  Jobs: ' + ', '.join(state['job_ids']) if state['job_ids'] else ''
    w_file_label.value = (
        f'✅  {len(state["loaded_files"])} file(s): {", ".join(state["loaded_files"])}'
        f'{job_str}  |  {len(dut_names)} DUT(s) total'
    )
    for btn in [w_btn_water, w_btn_steam, w_btn_table, w_btn_pdf]:
        btn.disabled = len(dut_names) == 0

def get_selected_duts(use_all_checkbox):
    if use_all_checkbox.value:
        return list(state['dut_data'].keys())
    return list(w_dut_select.value)

def on_plot_water(b):
    with out_plot:
        clear_output(wait=True)
        duts = get_selected_duts(w_select_all_water)
        if not duts:
            print('No DUTs selected.')
            return
        fig, _ = build_plot(duts, 'Water_L', 'Water Consumption per Cycle [L]',
                            'Water Consumption per Cycle',
                            w_thresh_water_min.value, w_thresh_water_max.value)
        plt.show()

def on_plot_steam(b):
    with out_plot:
        clear_output(wait=True)
        duts = get_selected_duts(w_select_all_steam)
        if not duts:
            print('No DUTs selected.')
            return
        fig, _ = build_plot(duts, 'Steam_gMin', 'Steam Rate per Cycle [g/min]',
                            'Steam Rate per Cycle',
                            w_thresh_steam_min.value, w_thresh_steam_max.value)
        plt.show()

def on_download_table(b):
    with out_table:
        clear_output(wait=True)
        if not state['dut_data']:
            print('No data loaded.')
            return
        try:
            print('⏳ Preparing XLSX …')
            df = make_full_cycle_table()
            job_part  = '_'.join(state['job_ids']) if state['job_ids'] else 'export'
            xlsx_name = f'{job_part}_summary_table.xlsx'
            buf = io.BytesIO()
            with pd.ExcelWriter(buf, engine='openpyxl') as writer:
                df.to_excel(writer, index=False, sheet_name='Summary')
            buf.seek(0)
            with open(xlsx_name, 'wb') as f:
                f.write(buf.read())
            files.download(xlsx_name)
            clear_output(wait=True)
            print(f'✅ Downloaded: {xlsx_name}')
        except Exception as e:
            clear_output(wait=True)
            print(f'❌ Export failed: {e}')

def on_export_pdf(b):
    with out_plot:
        clear_output(wait=True)
        print('⏳ Generating PDF …')
    try:
        fname     = pdf_filename()
        pdf_bytes = generate_pdf_bytes()
        with open(fname, 'wb') as f:
            f.write(pdf_bytes)
        files.download(fname)
        with out_plot:
            clear_output(wait=True)
            print(f'✅ PDF exported: {fname}')
    except Exception as e:
        with out_plot:
            clear_output(wait=True)
            print(f'❌ PDF export failed: {e}')

w_upload.observe(on_upload, names='value')
w_btn_water.on_click(on_plot_water)
w_btn_steam.on_click(on_plot_steam)
w_btn_table.on_click(on_download_table)
w_btn_pdf.on_click(on_export_pdf)

# ── Layout & Display ──────────────────────────────────────────────────────────
header = widgets.HTML('<h3 style="color:#2c5f8a">🧪 Steam Iron Test Bench — Data Evaluation Tool</h3>')

box_upload = widgets.VBox([
    widgets.HTML('<b>Step 1 — Load XLSX files (max. 2)</b>'),
    w_upload,
    widgets.HBox([w_progress, w_progress_label]),
    w_file_label,
    out_status,
], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='6px', width='660px'))

box_dut = widgets.VBox([
    widgets.HTML('<b>Step 2 — Select DUTs</b><br><small>(used when "Select all" is unchecked)</small>'),
    w_dut_select,
], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='6px', width='300px'))

box_thresh = widgets.VBox([
    widgets.HTML('<b>Step 3 — Thresholds</b>'),
    widgets.HTML('<i>Water [L]:</i>'),
    w_thresh_water_min, w_thresh_water_max,
    widgets.HTML('<br><i>Steam [g/min]:</i>'),
    w_thresh_steam_min, w_thresh_steam_max,
], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='6px', width='300px'))

box_plots = widgets.VBox([
    widgets.HTML('<b>Step 4 — Plot & Evaluate</b>'),
    widgets.HBox([w_select_all_water, widgets.Label('  ← Water plot')]),
    w_btn_water,
    widgets.HTML('<br>'),
    widgets.HBox([w_select_all_steam, widgets.Label('  ← Steam plot')]),
    w_btn_steam,
    widgets.HTML('<br>'),
    w_btn_table,
    widgets.HTML('<br>'),
    w_btn_pdf,
], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='6px', width='380px'))

display(header, box_upload,
        widgets.HBox([box_dut, box_thresh, box_plots]),
        out_plot, out_table)